# 03 · Evaluation on 2024 (out-of-time test)

1. Predictions on the test year
2. Compare every model
3. Calibration
4. What LightGBM uses
5. Fairness check: with vs. without gender
6. Risk tiers
7. Final model choice

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make `src` importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src import config

from src.train import DISPLAY_NAMES, fit_all, usable_rows
from src.predict import predict_frame
from src.evaluate import (
    calibration_table, choose_model, comparison_table, lift_table, assign_risk_tier,
)

In [ ]:
train = pd.read_csv(config.TRAIN_FILE)
test = usable_rows(pd.read_csv(config.TEST_FILE))

models = fit_all(train)
predictions = {name: predict_frame(name, model, test) for name, model in models.items()}
y = test[config.TARGET]
p_glm = predictions["glm"]
p_lgbm = predictions["lgbm"]

## 2. Compare the models

AUC and Gini measure ranking. A higher score means the model puts policies that claimed above policies that did not. The calibration gap is the average distance between the predicted rate and the observed claim rate. Smaller is better. The claim rate on this test set is about 15%.

In [ ]:
scores = comparison_table(y, predictions)
print("Chosen model:", DISPLAY_NAMES[choose_model(scores)])
scores.rename(index=DISPLAY_NAMES).round(3)

In [ ]:
lift_table(y, p_lgbm, test[config.EXPOSURE_COL])

## 3. Calibration

The app shows probabilities to users, so they need to be correct, not just well ranked.

In [ ]:
for label, pred in [("GLM", p_glm), ("LightGBM", p_lgbm)]:
    cal = calibration_table(y, pred)
    plt.plot(cal["predicted_rate"], cal["actual_rate"], "o-", label=label)
lim = [0, 0.35]
plt.plot(lim, lim, "--", color="grey", label="Perfect calibration")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed claim rate")
plt.legend()

## 4. What LightGBM uses

The share below is how much each input contributed to the LightGBM splits, measured by gain. Exposure dominates because a policy that was in force for a short time has less chance to claim. Once the app fixes exposure at one full year, the remaining order is vehicle value, vehicle age, then driver age.

In [ ]:
names = config.FEATURES + [config.EXPOSURE_COL]
gain = pd.Series(
    models["lgbm"].named_steps["model"].booster_.feature_importance(importance_type="gain"),
    index=names,
)
(gain / gain.sum()).sort_values(ascending=False).rename("share of splits").round(3)

## 5. Fairness check

Since 2012, EU insurers cannot use gender to set prices. TODO: refit with and without gender (if available). How much does performance change? Does removing it shift predictions for other groups (proxy effects)?

## 6. Risk tiers

Low, Medium, and High are the lower, middle, and upper third of LightGBM scores on these 2024 policies. The app uses the same idea on a full next year, which is why `RISK_TIER_CUTOFFS` in `src/config.py` sit at about 16.6% and 19.6%.

In [ ]:
low, medium = np.quantile(p_lgbm, [1 / 3, 2 / 3])
tiers = assign_risk_tier(p_lgbm, {"Low": float(low), "Medium": float(medium)})
pd.DataFrame({"tier": tiers, "actual": y}).groupby("tier").agg(
    policies=("actual", "size"), actual_claim_rate=("actual", "mean"))

## 7. Final model choice

The app uses LightGBM.

On 2024, LightGBM has the best ranking and the smallest calibration gap: AUC 0.655, Gini 0.310, calibration gap 0.003. Random forest and XGBoost tie at AUC 0.648. The GLM is 0.643, logistic regression 0.642, and the single decision tree 0.638. The rule keeps a simpler model only when it is within 0.005 AUC of the leader. LightGBM's lead is 0.007, so it wins.

LightGBM adds hundreds of small trees. Each new tree corrects the policies the earlier trees scored too high or too low. Exposure accounts for about 71% of the splits, vehicle value 15%, vehicle age 8%, and driver age 6%. The GLM remains the model to read when you want a single percentage effect for vehicle value or vehicle age.